# 01 - Data Cleaning and Merging
The purpose of this notebook is to load, clean, and, merge the raw workout and bodyweight datasets. The primary steps include standardizing data types, handling missing values, and combining the two datasets on the "Date" column. The final output is a clean, master DataFrame that will be used for subsequent in-depth analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', 150)  # display up to 150 rows
pd.set_option('display.max_colwidth', None)  # increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # format floats to 2 decimal places 

In [ ]:
# loading csv files
workouts_df = pd.read_csv('../data/workout_data.csv')
weight_df = pd.read_csv('../data/measurement_data.csv')

### 2.1 - Previewing the Workout Data
To get a better idea of the way this data is organized, let's view the head along with any attributes with null values. This will help me decide what data I can safely delete.

In [242]:
workouts_df.head()  # preview the data

,title,start_time,end_time,exercise_title,set_index,set_type,weight_lbs,reps
0,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",Single Leg Press (Machine),0,normal,50.00,8.00
1,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",Single Leg Press (Machine),1,normal,80.00,4.00
2,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",Leg Press Horizontal (Machine),0,normal,200.00,8.00
3,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",Leg Press Horizontal (Machine),1,normal,250.00,8.00
4,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",Seated Leg Curl (Machine),0,warmup,50.00,8.00


In [241]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           2042 non-null   object 
 1   start_time      2042 non-null   object 
 2   end_time        2042 non-null   object 
 3   exercise_title  2042 non-null   object 
 4   set_index       2042 non-null   int64  
 5   set_type        2042 non-null   object 
 6   weight_lbs      1957 non-null   float64
 7   reps            2042 non-null   float64
dtypes: float64(2), int64(1), object(5)
memory usage: 143.6+ KB


In [ ]:
workouts_df.isnull().sum()  # check how many null values are present for each column

### 2.2 - Removing Workout Metadata
We can immediately see some metadata that will not be necessary in our data analysis, so we can remove them. This includes the description and exercise notes. The title of the workout itself is also not strictly necessary in the data analysis, but may make it easier to organize specific exercises done on each day into their respective muscle groups.

In [ ]:
workouts_df = workouts_df.drop(['description', 'exercise_notes'], axis=1)  # remove the listed columns

### 2.3 - Verifying Irrelevant Exercise Data
Since we aim to look at strength training data, we can get rid of any data related to cardio or other exercise types. These exercises will not have weight or rep information. However, some exercises, like pull ups, may not have weight information since it is based on bodyweight. These exercises are vital for checking trends in strength so they should not be deleted. Thus, we will analyze the data closer to verify what data should and should not be deleted.

In [ ]:
workouts_df[workouts_df['weight_lbs'].isnull()]  # select the rows without any value for weight_lbs, but without permanently deleting the rest from the df

In [ ]:
workouts_df[workouts_df['reps'].isnull()]

### 2.4 - Deleting Irrelevant Exercise Data
Since I have verified that bodyweight exercises do not have weight information, I will remove exercises based on other attributes, like distance and duration. Then, we can check back and make sure that there are no remaining irrelevant exercises in the dataset and look at the full datasets data types using info().

In [ ]:
workouts_df[workouts_df['distance_miles'].notna()]  # select only the rows with a value for distance_miles, without permanently deleting the rest

In [ ]:
workouts_df[workouts_df['duration_seconds'].notna()]

In [ ]:
# update the df to only keep rows where distance_miles and duration_seconds are null
workouts_df = workouts_df[workouts_df['distance_miles'].isnull()]
workouts_df = workouts_df[workouts_df['duration_seconds'].isnull()]

In [ ]:
workouts_df[workouts_df['reps'].isnull()]

In [ ]:
workouts_df.info()

### 2.5 - Final Removal of Irrelevant Workout Information
Since all our cardio workouts have been removed from the dataset, we can also remove the columns that track distance and duration. Additionally, since it is not particularly pertinent to this analysis, we can also get rid of superset information and RPE.

In [ ]:
workouts_df = workouts_df.drop(['superset_id', 'distance_miles', 'duration_seconds', 'rpe'], axis=1)

In [ ]:
workouts_df.info()

### 2.6 - Converting Workout Date Information to Datetime
To ensure that merging the datasets is a smooth process, we need to ensure that both date columns are datetime objects. However, there is a start time and an end time, something that will not be present on bodyweight data. We can keep and convert both to datetime and then later determine the workout duration if necessary for our analysis. Then we can use info() to ensure the correct data type is being used.

In [243]:
workouts_df['start_time'] = pd.to_datetime(workouts_df['start_time'])
workouts_df['end_time'] = pd.to_datetime(workouts_df['end_time'])

In [244]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           2042 non-null   object        
 1   start_time      2042 non-null   datetime64[ns]
 2   end_time        2042 non-null   datetime64[ns]
 3   exercise_title  2042 non-null   object        
 4   set_index       2042 non-null   int64         
 5   set_type        2042 non-null   object        
 6   weight_lbs      1957 non-null   float64       
 7   reps            2042 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(3)
memory usage: 143.6+ KB


### 2.7 - Previewing the Bodyweight Data
Now we will do much of the same data processing on the bodyweight dataset that we just did to the workout dataset. Then, after cleaning this dataset, we will be ready to merge the two datasets into the master DataFrame to be used for analysis.

In [ ]:
weight_df.head()

In [240]:
weight_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        763 non-null    object 
 1   weight_lbs  763 non-null    float64
dtypes: float64(1), object(1)
memory usage: 12.1+ KB


In [ ]:
weight_df.isnull().sum()

### 2.8 - Removing Empty Measurement Columns
As we can see, most of the columns are fully empty since I only tracked my bodyweight and the date. Therefore, we can remove all the other columns, leaving only what we need.

In [ ]:
weight_df = weight_df[['date', 'weight_lbs']]

In [ ]:
weight_df.info()

### 2.9 - Converting Bodyweight Date Information to Datetime